In [19]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
)
from sklearn.calibration import CalibrationDisplay

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_DIR = '../data/raw/'           # adjust to your Kaggle data path
GENDER = 'M'                 # 'M' for mens, 'W' for womens
VAL_START_SEASON = 2010      # first season used as a validation fold

import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)

from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid

In [20]:

EXPERIMENT_NAME = f'march_madness_{GENDER.lower()}_2026_v2'

mlflow.set_tracking_uri(f'../mlruns')
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow experiment: {EXPERIMENT_NAME}')

MLflow experiment: march_madness_m_2026_v2


In [21]:
prefix = GENDER  # 'M' or 'W'

tourney    = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneyCompactResults.csv')
seeds      = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneySeeds.csv')
reg_season = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonCompactResults.csv')

try:
    reg_detailed = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonDetailedResults.csv')
    print(f'Detailed box scores: {reg_detailed.Season.min()}–{reg_detailed.Season.max()}')
except FileNotFoundError:
    reg_detailed = None
    print('No detailed results file — basic features only.')

try:
    massey = pd.read_csv(f'{DATA_DIR}{prefix}MasseyOrdinals.csv')
    print(f'Massey ordinals: {massey.Season.min()}–{massey.Season.max()}')
except FileNotFoundError:
    massey = None
    print('No Massey ordinals file found.')

print(f'Tourney games: {len(tourney)}')
print(f'Seasons: {tourney.Season.min()}–{tourney.Season.max()}')

Detailed box scores: 2003–2026
Massey ordinals: 2003–2026
Tourney games: 2585
Seasons: 1985–2025


In [ ]:
# ── 4a. Parse seed number from seed string (e.g. 'W01' → 1) ──────────────────
def parse_seed(seed_str):
    return int(''.join(filter(str.isdigit, str(seed_str))))

seeds['SeedNum'] = seeds['Seed'].apply(parse_seed)
seed_lookup = seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()


# ── 4b. Elo ratings (carries across seasons with mean reversion) ──────────────
def compute_elo(results_df, k=32, initial=1500, carry_factor=0.75, use_mov=True):
    elo = {}
    season_end = {}

    for season, grp in results_df.sort_values(['Season', 'DayNum']).groupby('Season'):
        elo = {
            team: carry_factor * rating + (1 - carry_factor) * initial
            for team, rating in elo.items()
        }

        for _, row in grp.iterrows():
            w, l = row['WTeamID'], row['LTeamID']
            r_w = elo.get(w, initial)
            r_l = elo.get(l, initial)

            # Home court adjustment
            loc = row.get('WLoc', 'N')
            hca = 100
            r_w_adj = r_w + (hca if loc == 'H' else -hca if loc == 'A' else 0)

            exp_w = 1 / (1 + 10 ** ((r_l - r_w_adj) / 400))
            exp_l = 1 - exp_w

            # MOV adjusted K
            if use_mov and 'WScore' in row.index:
                mov = row['WScore'] - row['LScore']
                elo_diff = abs(r_w_adj - r_l)
                mov_mult = np.log(abs(mov) + 1.0)
                elo_mult = 2.2 / ((elo_diff * 0.001) + 2.2)
                k_adj = k * mov_mult * elo_mult
            else:
                k_adj = k

            elo[w] = r_w + k_adj * (1 - exp_w)
            elo[l] = r_l + k_adj * (0 - exp_l)

        for team, rating in elo.items():
            season_end[(season, team)] = rating

    return season_end

print('Computing Elo ratings...')
elo_ratings = compute_elo(reg_season, k=20, carry_factor=0.75)
print(f'Elo ratings computed for {len(elo_ratings)} (season, team) pairs.')

# ── 4c. Last-N form features from detailed box scores ────────────────────────
def compute_form_features(detailed_df, n_games=10):
    if detailed_df is None:
        return pd.DataFrame()

    records = []
    for _, row in detailed_df.iterrows():
        season = row['Season']
        day    = row['DayNum']

        w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
        l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
        poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

        w_off_eff = (row['WScore'] / poss) * 100
        l_off_eff = (row['LScore'] / poss) * 100

        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['WTeamID'], 'Win': 1,
            'Margin': row['WScore'] - row['LScore'],
            'OffEff': w_off_eff, 'DefEff': l_off_eff,
            'Pace': poss
        })
        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['LTeamID'], 'Win': 0,
            'Margin': row['LScore'] - row['WScore'],
            'OffEff': l_off_eff, 'DefEff': w_off_eff,
            'Pace': poss
        })

    game_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    form_rows = []
    for (season, team), grp in game_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)

        # ── Season-long stats ────────────────────────────────────────────────
        season_row = {
            'Season': season,
            'TeamID': team,
            # Season long
            'Season_WinRate':   grp['Win'].mean(),
            'Season_AvgMargin': grp['Margin'].mean(),
            'Season_AvgOffEff': grp['OffEff'].mean(),
            'Season_AvgDefEff': grp['DefEff'].mean(),
            'Season_EffMargin': grp['OffEff'].mean() - grp['DefEff'].mean(),
            'Season_AvgPace':   grp['Pace'].mean(),
            'Season_NumGames':  len(grp),
            # Last N
            f'Last{n_games}_WinRate':   last_n['Win'].mean(),
            f'Last{n_games}_AvgMargin': last_n['Margin'].mean(),
            f'Last{n_games}_AvgOffEff': last_n['OffEff'].mean(),
            f'Last{n_games}_AvgDefEff': last_n['DefEff'].mean(),
            f'Last{n_games}_EffMargin': last_n['OffEff'].mean() - last_n['DefEff'].mean(),
            f'Last{n_games}_AvgPace':   last_n['Pace'].mean(),
            # Trend — last N minus season average, captures peaking/declining
            'Trend_EffMargin': (last_n['OffEff'].mean() - last_n['DefEff'].mean()) - 
                               (grp['OffEff'].mean() - grp['DefEff'].mean()),
            'Trend_WinRate':   last_n['Win'].mean() - grp['Win'].mean(),
        }
        form_rows.append(season_row)

    return pd.DataFrame(form_rows).set_index(['Season', 'TeamID'])

print('Computing form features...')
form_features = compute_form_features(reg_detailed, n_games=10)
if not form_features.empty:
    print(f'Form features: {form_features.shape}')
else:
    print('No form features (detailed data unavailable).')


# ── 4d. Massey ordinal snapshot (as of Selection Sunday ~day 133) ─────────────
def get_massey_snapshot(massey_df, system='POM', day_cutoff=133):
    """
    Pull each team's most recent ranking before Selection Sunday.
    Defaults to KenPom (POM).
    """
    if massey_df is None:
        return {}

    filtered = massey_df[massey_df['RankingDayNum'] <= day_cutoff]

    if system in filtered['SystemName'].unique():
        filtered = filtered[filtered['SystemName'] == system]
    else:
        print(f'System {system} not found — using all systems.')

    latest = (
        filtered.sort_values('RankingDayNum')
        .groupby(['Season', 'TeamID'])
        .last()
        .reset_index()
    )

    return latest.set_index(['Season', 'TeamID'])['OrdinalRank'].to_dict()

massey_lookup = get_massey_snapshot(massey, system='POM')
print(f'Massey lookup entries: {len(massey_lookup)}')

coaches = pd.read_csv(f'{DATA_DIR}{prefix}TeamCoaches.csv')
tourney_compact = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneyCompactResults.csv')

def compute_coach_features(coaches_df, tourney_df):
    # coaches_df already has one row per (Season, TeamID, CoachName)
    # no need to expand — just deduplicate to be safe
    coach_df = coaches_df[['Season', 'TeamID', 'CoachName']].drop_duplicates(
        ['Season', 'TeamID']  # keep first coach if mid-season change
    )

    w_games = tourney_df[['Season', 'WTeamID']].copy()
    w_games.columns = ['Season', 'TeamID']
    w_games['TourneyWin'] = 1

    l_games = tourney_df[['Season', 'LTeamID']].copy()
    l_games.columns = ['Season', 'TeamID']
    l_games['TourneyWin'] = 0

    tourney_games = pd.concat([w_games, l_games])
    tourney_games = tourney_games.merge(coach_df, on=['Season', 'TeamID'], how='left')

    coach_stats = []
    for coach, coach_group in tourney_games.groupby('CoachName'):
        cumulative_wins        = 0
        cumulative_games       = 0
        cumulative_appearances = 0

        for season, season_group in coach_group.groupby('Season'):
            coach_stats.append({
                'Season':               season,
                'CoachName':            coach,
                'Coach_TourneyApps':    cumulative_appearances,
                'Coach_TourneyWins':    cumulative_wins,
                'Coach_TourneyGames':   cumulative_games,
                'Coach_TourneyWinRate': cumulative_wins / cumulative_games
                                        if cumulative_games > 0 else np.nan,
            })
            cumulative_wins        += season_group['TourneyWin'].sum()
            cumulative_games       += len(season_group)
            cumulative_appearances += 1

    coach_stats_df = pd.DataFrame(coach_stats)
    result = coach_df.merge(coach_stats_df, on=['Season', 'CoachName'], how='left')
    return result.set_index(['Season', 'TeamID'])

print('Computing coach features...')
coach_features = compute_coach_features(coaches, tourney)
print(f'Coach features: {coach_features.shape}')

Computing Elo ratings...
Elo ratings computed for 14206 (season, team) pairs.
Computing form features...
Form features: (8346, 15)
Massey lookup entries: 8352
Computing coach features...
Coach features: (13763, 5)


In [23]:
# Compute opponent win rate lookup — used inside compute_sos_features
win_counts = reg_season.groupby(['Season', 'WTeamID']).size().reset_index()
win_counts.columns = ['Season', 'TeamID', 'Wins']

game_counts = pd.concat([
    reg_season[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'}),
    reg_season[['Season', 'LTeamID']].rename(columns={'LTeamID': 'TeamID'})
]).groupby(['Season', 'TeamID']).size().reset_index()
game_counts.columns = ['Season', 'TeamID', 'Games']

win_rates = win_counts.merge(game_counts, on=['Season', 'TeamID'])
win_rates['WinRate'] = win_rates['Wins'] / win_rates['Games']
win_rate_lookup = win_rates.set_index(['Season', 'TeamID'])['WinRate'].to_dict()

print(f'Win rate lookup: {len(win_rate_lookup)} entries')

Win rate lookup: 13736 entries


In [24]:
def compute_sos_features(results_df, elo_ratings, win_rate_lookup, detailed_df=None, n_games=10):
    """
    Compute strength of schedule features for each (season, team).
    
    Two versions:
    - Season SOS: average opponent Elo and efficiency margin across all regular season games
    - Last-N SOS: same but only for the last n games
    
    Returns DataFrame indexed by (Season, TeamID).
    """
    records = []
    for _, row in results_df.sort_values(['Season', 'DayNum']).iterrows():
        season = row['Season']
        w_team, l_team = row['WTeamID'], row['LTeamID']
        w_elo = elo_ratings.get((season, w_team), 1500)
        l_elo = elo_ratings.get((season, l_team), 1500)

        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': w_team, 'OppElo': l_elo,
            'OppWinRate': win_rate_lookup.get((season, l_team), 0.5),
        })
        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': l_team, 'OppElo': w_elo,
            'OppWinRate': win_rate_lookup.get((season, w_team), 0.5),
        })

    opp_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    # If detailed box scores available, also compute opponent efficiency margin
    if detailed_df is not None:
        # Reuse the game_df logic from compute_form_features
        eff_records = []
        for _, row in detailed_df.iterrows():
            season = row['Season']
            day    = row['DayNum']

            w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
            l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
            poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

            w_eff_margin = ((row['WScore'] - row['LScore']) / poss) * 100
            l_eff_margin = -w_eff_margin

            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['WTeamID'], 'OppEffMargin': l_eff_margin
            })
            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['LTeamID'], 'OppEffMargin': w_eff_margin
            })

        eff_df = pd.DataFrame(eff_records).sort_values(['Season', 'TeamID', 'DayNum'])
        opp_df = opp_df.merge(eff_df, on=['Season', 'DayNum', 'TeamID'], how='left')
    else:
        opp_df['OppEffMargin'] = np.nan

    # Compute season and last-N SOS per team
    sos_rows = []
    for (season, team), grp in opp_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)
        row = {
            'Season': season,
            'TeamID': team,
            # Season SOS
            'SOS_Season_AvgOppElo':       grp['OppElo'].mean(),
            'SOS_Season_AvgOppEffMargin': grp['OppEffMargin'].mean(),
            'SOS_Season_AvgOppWinRate':   grp['OppWinRate'].mean(),   # ← add this
            # Last-N SOS
            f'SOS_Last{n_games}_AvgOppElo':       last_n['OppElo'].mean(),
            f'SOS_Last{n_games}_AvgOppEffMargin': last_n['OppEffMargin'].mean(),
            f'SOS_Last{n_games}_AvgOppWinRate':   last_n['OppWinRate'].mean(),  # ← and this
        }
        sos_rows.append(row)

    return pd.DataFrame(sos_rows).set_index(['Season', 'TeamID'])


print('Computing SOS features...')
sos_features = compute_sos_features(reg_season, elo_ratings, win_rate_lookup, reg_detailed, n_games=10)
print(f'SOS features: {sos_features.shape}')

Computing SOS features...
SOS features: (13753, 6)


In [25]:
def build_matchup_features(tourney_df, elo_ratings, seed_lookup,
                            form_features, massey_lookup, coach_features, sos_features):
    """
    One row per tournament game with pairwise features.
    Team A is always the lower seed number (stronger team).
    Label: 1 if team A wins, 0 otherwise.
    """
    rows = []
    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_team, l_team = game['WTeamID'], game['LTeamID']

        w_seed = seed_lookup.get((season, w_team), 16)
        l_seed = seed_lookup.get((season, l_team), 16)

        if w_seed <= l_seed:
            t_a, t_b, label = w_team, l_team, 1
        else:
            t_a, t_b, label = l_team, w_team, 0

        seed_a   = seed_lookup.get((season, t_a), 16)
        seed_b   = seed_lookup.get((season, t_b), 16)
        elo_a    = elo_ratings.get((season, t_a), 1500)
        elo_b    = elo_ratings.get((season, t_b), 1500)
        massey_a = massey_lookup.get((season, t_a), np.nan)
        massey_b = massey_lookup.get((season, t_b), np.nan)

        row = {
            'Season': season, 'TeamA': t_a, 'TeamB': t_b, 'Label': label,
            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,
            'EloA': elo_a,   'EloB': elo_b,   'EloDiff':  elo_a - elo_b,
            # Massey: lower rank = better, so flip sign so positive = A is better
            'MasseyA': massey_a, 'MasseyB': massey_b,
            'MasseyDiff': (massey_b - massey_a)
                if not (np.isnan(massey_a) or np.isnan(massey_b)) else np.nan,
        }

        # Form features
        if not form_features.empty:
            for col in form_features.columns:
                val_a = form_features.loc[(season, t_a), col] \
                        if (season, t_a) in form_features.index else np.nan
                val_b = form_features.loc[(season, t_b), col] \
                        if (season, t_b) in form_features.index else np.nan
                row[f'{col}_A']    = val_a
                row[f'{col}_B']    = val_b
                row[f'{col}_Diff'] = val_a - val_b \
                        if not (np.isnan(val_a) or np.isnan(val_b)) else np.nan
        
        if not sos_features.empty:
            for col in sos_features.columns:
                val_a = sos_features.loc[(season, t_a), col] \
                        if (season, t_a) in sos_features.index else np.nan
                val_b = sos_features.loc[(season, t_b), col] \
                        if (season, t_b) in sos_features.index else np.nan
                row[f'{col}_A']    = val_a
                row[f'{col}_B']    = val_b
                row[f'{col}_Diff'] = val_a - val_b \
                        if not (np.isnan(val_a) or np.isnan(val_b)) else np.nan
        
        if not coach_features.empty:
            for col in ['Coach_TourneyApps', 'Coach_TourneyWins',
                        'Coach_TourneyWinRate', 'Coach_TourneyGames']:
                val_a = coach_features.loc[(season, t_a), col] \
                        if (season, t_a) in coach_features.index else np.nan
                val_b = coach_features.loc[(season, t_b), col] \
                        if (season, t_b) in coach_features.index else np.nan
                row[f'{col}_A']    = val_a
                row[f'{col}_B']    = val_b
                row[f'{col}_Diff'] = val_a - val_b \
                        if not (np.isnan(val_a) or np.isnan(val_b)) else np.nan
            
        rows.append(row)

    return pd.DataFrame(rows)

matchups = build_matchup_features(
    tourney, elo_ratings, seed_lookup, form_features, massey_lookup, coach_features, sos_features
)

print(f'Matchup dataset: {matchups.shape}')
print(f'Seasons: {matchups.Season.min()}–{matchups.Season.max()}')
print(f'Higher seed win rate: {matchups.Label.mean():.3f}')

Matchup dataset: (2585, 88)
Seasons: 1985–2025
Higher seed win rate: 0.727


In [26]:
def walk_forward_cv(matchups_df, feature_cols, val_start=VAL_START_SEASON,
                    impute=True, season_decay=None):
    SKIP_SEASONS = {2020}
    seasons     = sorted(matchups_df['Season'].unique())
    val_seasons = [s for s in seasons if s >= val_start and s not in SKIP_SEASONS]

    folds = []
    for val_season in val_seasons:
        train = matchups_df[matchups_df['Season'] <  val_season]
        val   = matchups_df[matchups_df['Season'] == val_season]

        if len(train) == 0 or len(val) == 0:
            continue

        y_train = train['Label']
        y_val   = val['Label']

        if impute:
            train_median = train[feature_cols].median()
            X_train = train[feature_cols].fillna(train_median)
            X_val   = val[feature_cols].fillna(train_median)
        else:
            X_train = train[feature_cols]
            X_val   = val[feature_cols]

        # Sample weights — exponential decay by season
        if season_decay is not None:
            max_season   = train['Season'].max()
            sample_weights = season_decay ** (max_season - train['Season'])
            sample_weights = sample_weights / sample_weights.sum() * len(train)
        else:
            sample_weights = None

        folds.append((val_season, X_train, y_train, X_val, y_val, sample_weights))

    return folds

In [27]:
def compute_metrics(y_true, y_pred_proba):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred_proba)

    return {
        'brier_score': brier_score_loss(y_true, y_pred),
        'log_loss':    log_loss(y_true, y_pred),
        'roc_auc':     roc_auc_score(y_true, y_pred),
        'accuracy':    accuracy_score(y_true, (y_pred >= 0.5).astype(int)),
        'mean_pred':   y_pred.mean(),
        'n_games':     len(y_true),
    }


def plot_calibration(all_y_true, all_y_pred, run_name):
    fig, ax = plt.subplots(figsize=(6, 5))
    CalibrationDisplay.from_predictions(
        np.concatenate(all_y_true),
        np.concatenate(all_y_pred),
        n_bins=10, ax=ax, name=run_name
    )
    ax.set_title(f'Calibration — {run_name}')
    plt.tight_layout()
    return fig

In [28]:
def run_experiment(run_name, model, feature_cols, matchups_df,
                   tags=None, val_start=VAL_START_SEASON, scale=True, impute=True):
    steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    pipeline = Pipeline(steps)

    folds = walk_forward_cv(matchups_df, feature_cols, val_start=val_start, impute=impute)

    fold_metrics = []
    all_y_true, all_y_pred = [], []

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param('features',         feature_cols)
        mlflow.log_param('n_features',       len(feature_cols))
        mlflow.log_param('model_type',       type(model).__name__)
        mlflow.log_param('val_start_season', val_start)
        mlflow.log_param('n_folds',          len(folds))
        mlflow.log_param('gender',           GENDER)
        mlflow.log_param('scale',            scale)

        if hasattr(model, 'get_params'):
            mlflow.log_params({
                f'model__{k}': v for k, v in model.get_params().items()
            })
        if tags:
            mlflow.set_tags(tags)

        for val_season, X_train, y_train, X_val, y_val, sample_weights in folds:
            fit_params = {}
            if sample_weights is not None:
                fit_params['model__sample_weight'] = sample_weights

            pipeline.fit(X_train, y_train, **fit_params)
            y_pred = pipeline.predict_proba(X_val)[:, 1]
            
            metrics = compute_metrics(y_val, y_pred)
            metrics['season'] = val_season
            fold_metrics.append(metrics)
            all_y_true.append(y_val.values)
            all_y_pred.append(y_pred)

            mlflow.log_metrics({
                f'fold_{val_season}_brier':    metrics['brier_score'],
                f'fold_{val_season}_logloss':  metrics['log_loss'],
                f'fold_{val_season}_auc':      metrics['roc_auc'],
                f'fold_{val_season}_accuracy': metrics['accuracy'],
            })

        metrics_df = pd.DataFrame(fold_metrics)
        agg = {
            'mean_brier':    metrics_df['brier_score'].mean(),
            'std_brier':     metrics_df['brier_score'].std(),
            'min_brier':     metrics_df['brier_score'].min(),
            'max_brier':     metrics_df['brier_score'].max(),
            'mean_logloss':  metrics_df['log_loss'].mean(),
            'mean_auc':      metrics_df['roc_auc'].mean(),
            'mean_accuracy': metrics_df['accuracy'].mean(),
        }
        modern_folds = metrics_df[metrics_df['season'] >= 2021]
        agg['modern_mean_brier'] = modern_folds['brier_score'].mean()
        agg['modern_std_brier']  = modern_folds['brier_score'].std()
        agg['modern_mean_auc']   = modern_folds['roc_auc'].mean()
        mlflow.log_metrics(agg)

        fig = plot_calibration(all_y_true, all_y_pred, run_name)
        mlflow.log_figure(fig, 'calibration_curve.png')
        plt.close(fig)

        X_all = matchups_df[feature_cols].fillna(matchups_df[feature_cols].median())
        y_all = matchups_df['Label']
        pipeline.fit(X_all, y_all)
        mlflow.sklearn.log_model(pipeline, 'model')

    print(f'[{run_name}]')
    print(f'  Mean Brier (2013–2025) : {agg["mean_brier"]:.4f} ± {agg["std_brier"]:.4f}')
    print(f'  Mean Brier (2021–2025) : {agg["modern_mean_brier"]:.4f} ± {agg["modern_std_brier"]:.4f}')
    print(f'  Mean LogLoss           : {agg["mean_logloss"]:.4f}')
    print(f'  Mean AUC               : {agg["mean_auc"]:.4f}')
    print(f'  Mean Acc               : {agg["mean_accuracy"]:.4f}')
    print()

    return metrics_df, pipeline

In [29]:
features_baseline = [
    # Seeds
    'SeedA', 'SeedB', 'SeedDiff',
    # Elo
    'EloA', 'EloB', 'EloDiff',
    # Massey (raw values may not exist if you only stored diff — check matchups.columns)
    'MasseyDiff',
]

# Filter to only columns that exist in matchups
features_baseline = [f for f in features_baseline if f in matchups.columns]
print(f'Baseline features ({len(features_baseline)}): {features_baseline}')

Baseline features (7): ['SeedA', 'SeedB', 'SeedDiff', 'EloA', 'EloB', 'EloDiff', 'MasseyDiff']


In [30]:
param_grid_refined = {
    'n_estimators':     [50, 75, 100],
    'max_depth':        [2],
    'learning_rate':    [0.06, 0.07, 0.08],
    'subsample':        [0.5, 0.6],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [3],
    'gamma':            [0],
}

for params in ParameterGrid(param_grid_refined):
    run_name = (f'xgb3_n{params["n_estimators"]}_lr{params["learning_rate"]}'
                f'_ss{params["subsample"]}_cs{params["colsample_bytree"]}')
    run_experiment(
        run_name=run_name,
        model=XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=RANDOM_SEED,
            verbosity=0,
            **params
        ),
        feature_cols=features_baseline,
        matchups_df=matchups,
        val_start=2013,
        scale=False,
        impute=False,
        tags={
            'stage':      'xgb3_baseline_sweep',
            'gender':     GENDER,
            'elo_k':      '32',
            'elo_carry':  '0.85',
            'features':   'baseline'
        }
    )

print('Sweep complete.')

[xgb3_n50_lr0.06_ss0.5_cs0.6]
  Mean Brier (2013–2025) : 0.1771 ± 0.0246
  Mean Brier (2021–2025) : 0.1806 ± 0.0325
  Mean LogLoss           : 0.5290
  Mean AUC               : 0.7135
  Mean Acc               : 0.7335

[xgb3_n50_lr0.06_ss0.6_cs0.6]
  Mean Brier (2013–2025) : 0.1767 ± 0.0243
  Mean Brier (2021–2025) : 0.1793 ± 0.0322
  Mean LogLoss           : 0.5277
  Mean AUC               : 0.7205
  Mean Acc               : 0.7310

[xgb3_n75_lr0.06_ss0.5_cs0.6]
  Mean Brier (2013–2025) : 0.1771 ± 0.0256
  Mean Brier (2021–2025) : 0.1819 ± 0.0345
  Mean LogLoss           : 0.5284
  Mean AUC               : 0.7131
  Mean Acc               : 0.7347

[xgb3_n75_lr0.06_ss0.6_cs0.6]
  Mean Brier (2013–2025) : 0.1764 ± 0.0255
  Mean Brier (2021–2025) : 0.1807 ± 0.0345
  Mean LogLoss           : 0.5266
  Mean AUC               : 0.7189
  Mean Acc               : 0.7323

[xgb3_n100_lr0.06_ss0.5_cs0.6]
  Mean Brier (2013–2025) : 0.1774 ± 0.0255
  Mean Brier (2021–2025) : 0.1831 ± 0.0331
  Mean 

In [31]:
BEST_PARAMS = dict(
    n_estimators=75,
    max_depth=2,
    learning_rate=0.06,   # ← was 0.07 previously
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_weight=3,
    gamma=0,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_SEED,
    verbosity=0,
)

In [32]:
features_baseline = [
    'SeedA', 'SeedB', 'SeedDiff',
    'EloA', 'EloB', 'EloDiff',
    'MasseyDiff',
]

features_with_season = features_baseline + [
    'Season_WinRate_A',    'Season_WinRate_B',    'Season_WinRate_Diff',
    'Season_AvgOffEff_A',  'Season_AvgOffEff_B',  'Season_AvgOffEff_Diff',
    'Season_AvgDefEff_A',  'Season_AvgDefEff_B',  'Season_AvgDefEff_Diff',
    'Season_EffMargin_A',  'Season_EffMargin_B',  'Season_EffMargin_Diff',
    'Season_AvgPace_A',    'Season_AvgPace_B',    'Season_AvgPace_Diff',
]

features_with_last10 = features_baseline + [
    'Last10_WinRate_A',    'Last10_WinRate_B',    'Last10_WinRate_Diff',
    'Last10_AvgOffEff_A',  'Last10_AvgOffEff_B',  'Last10_AvgOffEff_Diff',
    'Last10_AvgDefEff_A',  'Last10_AvgDefEff_B',  'Last10_AvgDefEff_Diff',
    'Last10_EffMargin_A',  'Last10_EffMargin_B',  'Last10_EffMargin_Diff',
]

features_with_trend = features_baseline + [
    'Trend_EffMargin_A',   'Trend_EffMargin_B',   'Trend_EffMargin_Diff',
    'Trend_WinRate_A',     'Trend_WinRate_B',     'Trend_WinRate_Diff',
]

features_with_sos = features_baseline + [
    'SOS_Season_AvgOppElo_A',          'SOS_Season_AvgOppElo_B',          'SOS_Season_AvgOppElo_Diff',
    'SOS_Season_AvgOppEffMargin_A',    'SOS_Season_AvgOppEffMargin_B',    'SOS_Season_AvgOppEffMargin_Diff',
    'SOS_Season_AvgOppWinRate_A',      'SOS_Season_AvgOppWinRate_B',      'SOS_Season_AvgOppWinRate_Diff',
    'SOS_Last10_AvgOppElo_A',          'SOS_Last10_AvgOppElo_B',          'SOS_Last10_AvgOppElo_Diff',
    'SOS_Last10_AvgOppEffMargin_A',    'SOS_Last10_AvgOppEffMargin_B',    'SOS_Last10_AvgOppEffMargin_Diff',
]

features_with_coach = features_baseline + [
    'Coach_TourneyApps_A',      'Coach_TourneyApps_B',
    'Coach_TourneyWins_A',      'Coach_TourneyWins_B',
    'Coach_TourneyWinRate_A',   'Coach_TourneyWinRate_B',   'Coach_TourneyWinRate_Diff',
]

features_full = [
    # Seeds
    'SeedA', 'SeedB', 'SeedDiff',
    # Elo
    'EloA', 'EloB', 'EloDiff',
    # Massey
    'MasseyDiff',
    # Season stats
    'Season_WinRate_A',    'Season_WinRate_B',    'Season_WinRate_Diff',
    'Season_AvgOffEff_A',  'Season_AvgOffEff_B',  'Season_AvgOffEff_Diff',
    'Season_AvgDefEff_A',  'Season_AvgDefEff_B',  'Season_AvgDefEff_Diff',
    'Season_EffMargin_A',  'Season_EffMargin_B',  'Season_EffMargin_Diff',
    'Season_AvgPace_A',    'Season_AvgPace_B',    'Season_AvgPace_Diff',
    # Last 10
    'Last10_WinRate_A',    'Last10_WinRate_B',    'Last10_WinRate_Diff',
    'Last10_AvgOffEff_A',  'Last10_AvgOffEff_B',  'Last10_AvgOffEff_Diff',
    'Last10_AvgDefEff_A',  'Last10_AvgDefEff_B',  'Last10_AvgDefEff_Diff',
    'Last10_EffMargin_A',  'Last10_EffMargin_B',  'Last10_EffMargin_Diff',
    # Trend
    'Trend_EffMargin_A',   'Trend_EffMargin_B',   'Trend_EffMargin_Diff',
    'Trend_WinRate_A',     'Trend_WinRate_B',     'Trend_WinRate_Diff',
    # SOS
    'SOS_Season_AvgOppElo_A',          'SOS_Season_AvgOppElo_B',          'SOS_Season_AvgOppElo_Diff',
    'SOS_Season_AvgOppEffMargin_A',    'SOS_Season_AvgOppEffMargin_B',    'SOS_Season_AvgOppEffMargin_Diff',
    'SOS_Season_AvgOppWinRate_A',      'SOS_Season_AvgOppWinRate_B',      'SOS_Season_AvgOppWinRate_Diff',
    'SOS_Last10_AvgOppElo_A',          'SOS_Last10_AvgOppElo_B',          'SOS_Last10_AvgOppElo_Diff',
    'SOS_Last10_AvgOppEffMargin_A',    'SOS_Last10_AvgOppEffMargin_B',    'SOS_Last10_AvgOppEffMargin_Diff',
    # Coach
    'Coach_TourneyApps_A',      'Coach_TourneyApps_B',
    'Coach_TourneyWins_A',      'Coach_TourneyWins_B',
    'Coach_TourneyWinRate_A',   'Coach_TourneyWinRate_B',   'Coach_TourneyWinRate_Diff',
]

# Filter to only cols that exist in matchups
def filter_features(feature_list, df):
    available = [f for f in feature_list if f in df.columns]
    missing   = [f for f in feature_list if f not in df.columns]
    if missing:
        print(f'Missing features: {missing}')
    return available


# ── Run feature experiments ───────────────────────────────────────────────────
experiments = {
    'xgb_feat_baseline':    features_baseline,
    'xgb_feat_season':      features_with_season,
    'xgb_feat_last10':      features_with_last10,
    'xgb_feat_trend':       features_with_trend,
    'xgb_feat_sos':         features_with_sos,
    'xgb_feat_coach':       features_with_coach,
    'xgb_feat_full':        features_full,
}

for run_name, feature_list in experiments.items():
    feats = filter_features(feature_list, matchups)
    run_experiment(
        run_name=f'v2_{run_name}',           # ← prefix to distinguish from first ablation
        model=XGBClassifier(**BEST_PARAMS),  # ← make sure BEST_PARAMS uses lr=0.06
        feature_cols=feats,
        matchups_df=matchups,
        val_start=2013,
        scale=False,
        impute=False,
        tags={
            'stage':        'feature_ablation_v2',  # ← updated
            'gender':       GENDER,
            'features':     run_name.replace('xgb_feat_', ''),
            'n_features':   str(len(feats)),
            'elo_k':        '32',                   # ← new
            'elo_carry':    '0.85',                 # ← new
        }
    )

[v2_xgb_feat_baseline]
  Mean Brier (2013–2025) : 0.1764 ± 0.0255
  Mean Brier (2021–2025) : 0.1807 ± 0.0345
  Mean LogLoss           : 0.5266
  Mean AUC               : 0.7189
  Mean Acc               : 0.7323

[v2_xgb_feat_season]
  Mean Brier (2013–2025) : 0.1781 ± 0.0271
  Mean Brier (2021–2025) : 0.1823 ± 0.0357
  Mean LogLoss           : 0.5335
  Mean AUC               : 0.7088
  Mean Acc               : 0.7335

[v2_xgb_feat_last10]
  Mean Brier (2013–2025) : 0.1766 ± 0.0259
  Mean Brier (2021–2025) : 0.1810 ± 0.0348
  Mean LogLoss           : 0.5290
  Mean AUC               : 0.7179
  Mean Acc               : 0.7322

[v2_xgb_feat_trend]
  Mean Brier (2013–2025) : 0.1766 ± 0.0262
  Mean Brier (2021–2025) : 0.1808 ± 0.0336
  Mean LogLoss           : 0.5290
  Mean AUC               : 0.7146
  Mean Acc               : 0.7335

[v2_xgb_feat_sos]
  Mean Brier (2013–2025) : 0.1762 ± 0.0259
  Mean Brier (2021–2025) : 0.1794 ± 0.0350
  Mean LogLoss           : 0.5279
  Mean AUC           

In [ ]:
param_grid_full = {
    'n_estimators':     [50, 75, 100],
    'max_depth':        [2],
    'learning_rate':    [0.05, 0.06, 0.07],
    'subsample':        [0.4, 0.5, 0.6],
    'colsample_bytree': [0.4, 0.5, 0.6],
    'min_child_weight': [3, 5, 7],
    'gamma':            [0],
}

feats_full = filter_features(features_full, matchups)
for params in ParameterGrid(param_grid_full):
    run_name = (f'xgb_full_n{params["n_estimators"]}_lr{params["learning_rate"]}'
                f'_ss{params["subsample"]}_cs{params["colsample_bytree"]}'
                f'_mcw{params["min_child_weight"]}')
    run_experiment(
        run_name=run_name,
        model=XGBClassifier(objective='binary:logistic',
                            eval_metric='logloss',
                            random_state=RANDOM_SEED,
                            verbosity=0, **params),
        feature_cols=feats_full,
        matchups_df=matchups,
        val_start=2013,
        scale=False,
        impute=False,
        tags={'stage': 'full_feature_sweep', 'gender': GENDER,
              'elo_k': '32', 'elo_carry': '0.85'}
    )

[xgb_full_n50_lr0.05_ss0.4_cs0.4_mcw3]
  Mean Brier (2013–2025) : 0.1780 ± 0.0252
  Mean Brier (2021–2025) : 0.1800 ± 0.0327
  Mean LogLoss           : 0.5330
  Mean AUC               : 0.7137
  Mean Acc               : 0.7297

[xgb_full_n50_lr0.05_ss0.5_cs0.4_mcw3]
  Mean Brier (2013–2025) : 0.1775 ± 0.0252
  Mean Brier (2021–2025) : 0.1803 ± 0.0339
  Mean LogLoss           : 0.5312
  Mean AUC               : 0.7128
  Mean Acc               : 0.7297

[xgb_full_n50_lr0.05_ss0.6_cs0.4_mcw3]
  Mean Brier (2013–2025) : 0.1775 ± 0.0246
  Mean Brier (2021–2025) : 0.1788 ± 0.0325
  Mean LogLoss           : 0.5312
  Mean AUC               : 0.7198
  Mean Acc               : 0.7310

[xgb_full_n75_lr0.05_ss0.4_cs0.4_mcw3]
  Mean Brier (2013–2025) : 0.1788 ± 0.0266
  Mean Brier (2021–2025) : 0.1819 ± 0.0353
  Mean LogLoss           : 0.5351
  Mean AUC               : 0.7050
  Mean Acc               : 0.7335

[xgb_full_n75_lr0.05_ss0.5_cs0.4_mcw3]
  Mean Brier (2013–2025) : 0.1777 ± 0.0265
  Mean